In [6]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

True

In [2]:
print("SAMPLE:", os.getenv("SAMPLE"))

SAMPLE: None


In [7]:

from openai import OpenAI
import os
client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

response = client.responses.create(
    # input="Explain GenAIOps in one line.",
    input="What is Mistral?",
    # model="openai/gpt-oss-20b",
    model="llama-3.3-70b-versatile",
)
print(response.output_text)


AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'expired_api_key'}}

In [ ]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

with client.responses.stream(
    model="openai/gpt-oss-20b",
    input="write a blog in 200 lines on topic GenAIOps",
) as stream:
    for event in stream:
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)

    print()  # newline after stream finishes
    final_response = stream.get_final_response()


Generative AI Ops: The Next Frontier in IT Operations.  
As enterprises grapple with digital transformation, the demand for smarter, faster, and autonomous IT operations has never been higher.  
Enter GenAIOps, a paradigm that blends generative AI with traditional AIOps to deliver predictive, self‑healing, and context‑aware services.  
In this blog, we unpack what GenAIOps is, why it matters, and how organizations can start building it.  
What Is AIOps?  
AIOps, or Artificial Intelligence for IT Operations, uses machine learning, data analytics, and automation to reduce complexity in IT.  
It excels at event correlation, anomaly detection, root‑cause analysis, and incident response.  
However, conventional AIOps is largely reactive, relying on rule sets and historical patterns.  
The generative twist changes that.  
What Is GenAIOps?  
GenAIOps adds generative AI—large language models, diffusion models, and multimodal transformers—to the AIOps stack.  
Instead of just detecting a fault

In [ ]:
import os
import sys # Unused import
from datetime import datetime

def my_function(name):
    # Rule 1: f-string without placeholders
    print(f"Hello world") 
    
    # Rule 2: Use of 'is' for literal comparison
    if name is "admin": 
        print("Welcome")

    # Rule 3: Missing whitespace/formatting issues
    x = [1,2,3,4] # No space after commas
    
    # Rule 4: Forgotten TODOs (if configured)
    # TODO: fix this later

# Rule 5: No newline at end of file

In [1]:
import os
import re
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)

MODEL = "openai/gpt-oss-20b"
temps = [0.0, 0.2, 0.8, 1.2]

SYSTEM = (
    "You are a strict text completer. "
    "Return only exactly 3 words that continue the sentence. "
    "No explanations, no quotes, no punctuation."
)

USER = "The capital of France is"

def extract_text(resp):
    if getattr(resp, "output_text", None):
        return resp.output_text.strip()
    parts = []
    for item in getattr(resp, "output", []) or []:
        for c in getattr(item, "content", []) or []:
            t = getattr(c, "text", None)
            if t:
                parts.append(t)
    return " ".join(parts).strip()

def normalize_3_words(text):
    words = re.findall(r"[A-Za-z']+", text)
    return " ".join(words[:3]) if words else "[EMPTY]"

for t in temps:
    print(f"\n=== Temperature {t} ===")
    for i in range(5):
        r = client.responses.create(
            model=MODEL,
            temperature=t,
            max_output_tokens=12,
            input=[
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": USER},
            ],
        )
        raw = extract_text(r)
        clean = normalize_3_words(raw)
        print(f"Run {i+1}: {clean}    (raw: {raw})")



=== Temperature 0.0 ===
Run 1: The user says    (raw: The user says: "The capital of France)
Run 2: The user says    (raw: The user says: "The capital of France)
Run 3: The user says    (raw: The user says: "The capital of France)
Run 4: The user says    (raw: The user says: "The capital of France)
Run 5: The user says    (raw: The user says: "The capital of France)

=== Temperature 0.2 ===
Run 1: The user says    (raw: The user says: "The capital of France)
Run 2: We need to    (raw: We need to respond with exactly 3 words)
Run 3: The user says    (raw: The user says: "The capital of France)
Run 4: We must output    (raw: We must output exactly 3 words that continue)
Run 5: The user says    (raw: The user says: "The capital of France)

=== Temperature 0.8 ===
Run 1: We must return    (raw: We must return exactly 3 words that continue)
Run 2: The user says    (raw: The user says: "The capital of France)
Run 3: The user The    (raw: The user: "The capital of France is)
Run 4: The user 

In [6]:
import os
from collections import defaultdict
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)

# Pick first available preferred model
preferred = [
    "llama-3.1-8b-instant",
    "llama-3.3-70b-versatile",
    "mixtral-8x7b-32768",
]
available = [m.id for m in client.models.list().data]
model = next((m for m in preferred if m in available), available[0])

temps = [0.0, 0.2, 0.8, 1.2]
runs_per_temp = 5

prompts = {
    "FACTUAL": "What is the capital of France? Answer in 3 words max.",
    "CREATIVE": "Write one vivid line about a dragon flying over Mumbai at sunrise.",
}

print(f"Using model: {model}\n")

for label, prompt in prompts.items():
    print("=" * 70)
    print(f"{label} PROMPT: {prompt}")
    print("=" * 70)

    summary = defaultdict(list)

    for t in temps:
        print(f"\n--- Temperature {t} ---")
        for i in range(runs_per_temp):
            r = client.chat.completions.create(
                model=model,
                temperature=t,
                top_p=1,
                max_tokens=50,
                messages=[{"role": "user", "content": prompt}],
            )
            text = (r.choices[0].message.content or "").strip().replace("\n", " ")
            summary[t].append(text)
            print(f"Run {i+1}: {text}")

    print("\nVariation summary (unique outputs out of 5):")
    for t in temps:
        uniq = len(set(summary[t]))
        print(f"Temp {t}: {uniq}/5 unique")
    print("\n")


Using model: llama-3.1-8b-instant

FACTUAL PROMPT: What is the capital of France? Answer in 3 words max.

--- Temperature 0.0 ---
Run 1: Paris is capital.
Run 2: Paris is capital.
Run 3: Paris is capital.
Run 4: Paris is capital.
Run 5: Paris is capital.

--- Temperature 0.2 ---
Run 1: Paris is capital.
Run 2: Paris is capital.
Run 3: Paris is capital.
Run 4: Paris is capital.
Run 5: Paris is capital.

--- Temperature 0.8 ---
Run 1: Paris is capital.
Run 2: Paris, France.
Run 3: Paris is the capital.
Run 4: Paris is the capital.
Run 5: Paris, France.

--- Temperature 1.2 ---
Run 1: Paris is capital.
Run 2: Paris is capital.
Run 3: Paris is capital.
Run 4: Paris is.
Run 5: Paris is capital.

Variation summary (unique outputs out of 5):
Temp 0.0: 1/5 unique
Temp 0.2: 1/5 unique
Temp 0.8: 3/5 unique
Temp 1.2: 2/5 unique


CREATIVE PROMPT: Write one vivid line about a dragon flying over Mumbai at sunrise.

--- Temperature 0.0 ---
Run 1: As the first rays of sunrise danced across the Arabia

#### Top p

In [7]:
import os
from collections import defaultdict
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)

preferred = ["llama-3.1-8b-instant", "llama-3.3-70b-versatile", "mixtral-8x7b-32768"]
available = [m.id for m in client.models.list().data]
model = next((m for m in preferred if m in available), available[0])

prompt = "Write one vivid line about a dragon flying over Mumbai at sunrise."
top_ps = [0.2, 0.5, 0.8, 1.0]
runs = 5

print("Model:", model)
summary = defaultdict(list)

for p in top_ps:
    print(f"\n=== top_p={p} (temperature fixed at 0.8) ===")
    for i in range(runs):
        r = client.chat.completions.create(
            model=model,
            temperature=0.8,
            top_p=p,
            max_tokens=50,
            messages=[{"role": "user", "content": prompt}],
        )
        text = (r.choices[0].message.content or "").strip().replace("\n", " ")
        summary[p].append(text)
        print(f"Run {i+1}: {text}")

print("\nUnique outputs:")
for p in top_ps:
    print(f"top_p={p}: {len(set(summary[p]))}/{runs}")


Model: llama-3.1-8b-instant

=== top_p=0.2 (temperature fixed at 0.8) ===
Run 1: As the first rays of sunrise danced across the Arabian Sea, a majestic dragon soared above the Mumbai skyline, its scales glinting like a thousand diamonds against the fiery hues of the dawn, its wings beating in powerful, rhythmic strokes that sent wis
Run 2: As the first rays of sunrise danced across the Arabian Sea, a majestic dragon soared above the Mumbai skyline, its scales glinting like a thousand diamonds against the fiery hues of the dawn, its wings beating in powerful, rhythmic strokes that sent wis
Run 3: As the first rays of sunrise danced across the Arabian Sea, a majestic dragon soared above the Mumbai skyline, its scales glinting like a thousand diamonds against the fiery hues of the dawn, its wings beating in powerful, rhythmic strokes that sent wis
Run 4: As the first rays of sunrise danced across the Mumbai skyline, a majestic dragon burst forth from the darkness, its scales glinting like